
# Agente 1 del TFM: interpretar consulta financiera y descargar datos con `yfinance`

Este notebook implementa una **primera versión operativa** del agente de entrada de tu arquitectura:

**mensaje del usuario → interpretación estructurada → resolución de ticker(s) → descarga con `yfinance` → exportación a CSV**

## Qué hace
- entiende consultas en lenguaje natural en español;
- intenta extraer:
  - activo(s) solicitados;
  - rango temporal (`start/end` o `period`);
  - `interval`;
- resuelve empresas, índices, materias primas, divisas, cripto y otros activos disponibles en Yahoo Finance;
- llama a `yf.download(...)`;
- guarda el resultado en un **CSV** con el formato de salida de `yfinance`.

## Idea de diseño
Esta implementación sigue la lógica del **agente especialista de interpretación + recuperación**:
1. **parser** de lenguaje natural;
2. **resolución de activos** (`Search` / `Lookup` de `yfinance` + alias conocidos);
3. **constructor de la petición**;
4. **descarga y exportación**.

> Nota: esta versión está pensada para ser **ejecutable y reproducible**. Más adelante puedes sustituir el parser heurístico por un LLM con salida estructurada si quieres acercarte todavía más al diseño multiagente final.


In [1]:
# Si te falta alguna dependencia, ejecuta esta celda.
# %pip install -q yfinance pandas python-dateutil pydantic

In [2]:

from __future__ import annotations

import json
import math
import re
import unicodedata
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import yfinance as yf
from dateutil.relativedelta import relativedelta
from pydantic import BaseModel, Field, model_validator


In [3]:

# =========================
# 1) Modelos de datos
# =========================

class ResolvedAsset(BaseModel):
    user_text: str
    ticker: str
    name: Optional[str] = None
    quote_type: Optional[str] = None
    exchange: Optional[str] = None
    source: str = "unknown"
    confidence: float = 0.0


class DownloadRequest(BaseModel):
    user_message: str
    asset_queries: list[str] = Field(default_factory=list)
    resolved_assets: list[ResolvedAsset] = Field(default_factory=list)

    start: Optional[str] = None
    end: Optional[str] = None
    period: Optional[str] = None
    interval: str = "1d"

    group_by: str = "ticker"
    auto_adjust: bool = False
    threads: bool = True
    progress: bool = False

    assumptions: list[str] = Field(default_factory=list)
    needs_clarification: bool = False

    @property
    def tickers(self) -> list[str]:
        return [a.ticker for a in self.resolved_assets]

    @model_validator(mode="after")
    def validate_dates(self):
        if self.period and (self.start or self.end):
            raise ValueError("Usa period o start/end, pero no ambos a la vez.")
        if not self.period and not self.start:
            raise ValueError("Falta rango temporal: usa period o start.")
        if not self.asset_queries:
            raise ValueError("No se ha detectado ningún activo en la consulta.")
        return self

    def to_download_kwargs(self) -> dict[str, Any]:
        kwargs = {
            "tickers": self.tickers,
            "interval": self.interval,
            "group_by": self.group_by,
            "auto_adjust": self.auto_adjust,
            "threads": self.threads,
            "progress": self.progress,
        }
        if self.period:
            kwargs["period"] = self.period
        else:
            kwargs["start"] = self.start
            kwargs["end"] = self.end
        return kwargs


# =========================
# 2) Utilidades de texto
# =========================

def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def normalize_text(text: str) -> str:
    return normalize_spaces(strip_accents(text.lower()))


def slugify(text: str) -> str:
    text = normalize_text(text)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")


def parse_iso_or_local_date(value: str) -> date:
    value = value.strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    raise ValueError(f"No se pudo interpretar la fecha: {value!r}")


def end_exclusive(d: date) -> str:
    return (d + timedelta(days=1)).isoformat()


def today_date() -> date:
    return date.today()


# =========================
# 3) Alias manuales útiles
# =========================
# Esto mejora mucho la experiencia con índices, materias primas, divisas y cripto.

ALIAS_TO_TICKER = {
    # Índices
    "sp500": "^GSPC",
    "s&p500": "^GSPC",
    "s&p 500": "^GSPC",
    "sp 500": "^GSPC",
    "standard and poor 500": "^GSPC",
    "nasdaq": "^IXIC",
    "nasdaq composite": "^IXIC",
    "nasdaq 100": "^NDX",
    "dow jones": "^DJI",
    "djia": "^DJI",
    "russell 2000": "^RUT",
    "ibex": "^IBEX",
    "ibex 35": "^IBEX",
    "euro stoxx 50": "^STOXX50E",
    "dax": "^GDAXI",
    "cac 40": "^FCHI",
    "nikkei": "^N225",
    "nikkei 225": "^N225",
    "ftse 100": "^FTSE",

    # Materias primas / futuros
    "oro": "GC=F",
    "gold": "GC=F",
    "plata": "SI=F",
    "silver": "SI=F",
    "petroleo": "CL=F",
    "petroleo wti": "CL=F",
    "crudo wti": "CL=F",
    "brent": "BZ=F",
    "gas natural": "NG=F",
    "cobre": "HG=F",
    "cacao": "CC=F",
    "cafe": "KC=F",
    "trigo": "ZW=F",
    "maiz": "ZC=F",
    "soja": "ZS=F",

    # FX
    "eurusd": "EURUSD=X",
    "eur/usd": "EURUSD=X",
    "euro dolar": "EURUSD=X",
    "euro dólar": "EURUSD=X",
    "usdjpy": "USDJPY=X",
    "usd/jpy": "USDJPY=X",
    "gbpusd": "GBPUSD=X",
    "gbp/usd": "GBPUSD=X",

    # Cripto
    "bitcoin": "BTC-USD",
    "btc": "BTC-USD",
    "ethereum": "ETH-USD",
    "eth": "ETH-USD",
    "solana": "SOL-USD",
    "xrp": "XRP-USD",
}


INTERVAL_MAP = {
    "1m": "1m",
    "2m": "2m",
    "5m": "5m",
    "15m": "15m",
    "30m": "30m",
    "60m": "60m",
    "90m": "90m",
    "1h": "1h",
    "1d": "1d",
    "5d": "5d",
    "1wk": "1wk",
    "1w": "1wk",
    "1mo": "1mo",
    "3mo": "3mo",
    "diario": "1d",
    "diaria": "1d",
    "daily": "1d",
    "semanal": "1wk",
    "semanales": "1wk",
    "weekly": "1wk",
    "mensual": "1mo",
    "mensuales": "1mo",
    "monthly": "1mo",
}

INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}


# =========================
# 4) Extracción de tiempo
# =========================

def parse_interval(message: str) -> Optional[str]:
    msg = normalize_text(message)

    # Primero los códigos exactos soportados
    exact_pattern = r"\b(1m|2m|5m|15m|30m|60m|90m|1h|1d|5d|1wk|1w|1mo|3mo)\b"
    m = re.search(exact_pattern, msg)
    if m:
        return INTERVAL_MAP[m.group(1)]

    # Después palabras más genéricas
    for key, value in INTERVAL_MAP.items():
        if re.search(rf"\b{re.escape(key)}\b", msg):
            return value

    return None


def parse_dates_or_period(message: str) -> tuple[Optional[str], Optional[str], Optional[str], list[str]]:
    """
    Devuelve: start, end, period, assumptions
    - Si puede usar fechas explícitas -> start/end
    - Si es un rango relativo o max -> period
    """
    msg = normalize_text(message)
    assumptions: list[str] = []

    # Casos tipo YTD / max
    if re.search(r"\b(ytd|year to date|ano hasta la fecha|año hasta la fecha|este ano|este año)\b", msg):
        start = date(today_date().year, 1, 1).isoformat()
        end = end_exclusive(today_date())
        assumptions.append("Se interpreta la consulta como YTD.")
        return start, end, None, assumptions

    if re.search(r"\b(max|maximo|maximo historico|maximo historial|todo el historico|todo el historico disponible)\b", msg):
        assumptions.append("Se usará period='max' para recuperar el histórico máximo disponible.")
        return None, None, "max", assumptions

    # Fechas explícitas
    date_matches = re.findall(r"\b(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2}|\d{2}-\d{2}-\d{4})\b", message)
    parsed_dates = []
    for value in date_matches:
        try:
            parsed_dates.append(parse_iso_or_local_date(value))
        except ValueError:
            pass

    if len(parsed_dates) >= 2:
        start = parsed_dates[0].isoformat()
        end = end_exclusive(parsed_dates[1])
        return start, end, None, assumptions

    # "desde 2020", "desde 2020-01-01"
    m_from_year = re.search(r"\bdesde\s+(20\d{2}|19\d{2})\b", msg)
    if m_from_year:
        start = date(int(m_from_year.group(1)), 1, 1).isoformat()

        m_to_year = re.search(r"\b(?:hasta|a)\s+(20\d{2}|19\d{2})\b", msg)
        if m_to_year:
            end = end_exclusive(date(int(m_to_year.group(1)), 12, 31))
        else:
            end = end_exclusive(today_date())

        return start, end, None, assumptions

    m_from_date = re.search(
        r"\bdesde\s+(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2}|\d{2}-\d{2}-\d{4})\b", message, flags=re.IGNORECASE
    )
    if m_from_date:
        start = parse_iso_or_local_date(m_from_date.group(1)).isoformat()

        m_to_date = re.search(
            r"\b(?:hasta|a)\s+(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2}|\d{2}-\d{2}-\d{4})\b", message, flags=re.IGNORECASE
        )
        if m_to_date:
            end = end_exclusive(parse_iso_or_local_date(m_to_date.group(1)))
        else:
            end = end_exclusive(today_date())

        return start, end, None, assumptions

    # Rangos relativos "en 5 anos", "ultimos 3 meses", "2 semanas", etc.
    relative_patterns = [
        r"\bultim[oa]s?\s+(\d+)\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
        r"\ben\s+(\d+)\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
        r"\bde\s+(\d+)\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
        r"\b(\d+)\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
    ]
    for pattern in relative_patterns:
        m = re.search(pattern, msg)
        if not m:
            continue

        n = int(m.group(1))
        unit = m.group(2)
        if unit.startswith("dia"):
            return None, None, f"{n}d", assumptions
        if unit.startswith("semana"):
            # yfinance no tiene '7d', así que semanas cortas las pasamos a start/end
            start = (today_date() - relativedelta(weeks=n)).isoformat()
            end = end_exclusive(today_date())
            assumptions.append("Las semanas se transforman a start/end para mantener compatibilidad.")
            return start, end, None, assumptions
        if unit.startswith("mes"):
            return None, None, f"{n}mo", assumptions
        if unit.startswith("ano") or unit.startswith("año"):
            return None, None, f"{n}y", assumptions

    # "desde 2020" ya cubierto. Si no hay rango temporal, ponemos 1y por defecto.
    assumptions.append("No se indicó rango temporal; se usa 1y por defecto.")
    return None, None, "1y", assumptions


def estimate_days(request: DownloadRequest) -> Optional[int]:
    if request.period:
        period = request.period.lower()
        if period == "max":
            return None
        m = re.fullmatch(r"(\d+)(d|mo|y)", period)
        if not m:
            return None
        n = int(m.group(1))
        unit = m.group(2)
        if unit == "d":
            return n
        if unit == "mo":
            return n * 30
        if unit == "y":
            return n * 365
        return None

    if request.start and request.end:
        start = datetime.strptime(request.start, "%Y-%m-%d").date()
        end_excl = datetime.strptime(request.end, "%Y-%m-%d").date()
        return (end_excl - start).days
    return None


def fix_interval_if_needed(interval: str, days: Optional[int]) -> tuple[str, Optional[str]]:
    """
    yfinance solo soporta intradía en los últimos ~60 días.
    Si el usuario pide un rango más largo con un intervalo intradía, se degrada a 1d.
    """
    if interval not in INTRADAY_INTERVALS:
        return interval, None
    if days is None:
        return "1d", "Se degradó el intervalo a 1d porque intradía no suele estar disponible para rangos muy largos."
    if days > 60:
        return "1d", "Se degradó el intervalo a 1d porque yfinance limita intradía a los últimos 60 días aproximadamente."
    return interval, None


# =========================
# 5) Extracción de activos
# =========================

COMMAND_PATTERNS = [
    r"\bcuanto ha crecido\b",
    r"\bcuanto subio\b",
    r"\bcual ha sido el rendimiento de\b",
    r"\bcual es el rendimiento de\b",
    r"\bdescargame\b",
    r"\bdescarga\b",
    r"\bquiero\b",
    r"\bdame\b",
    r"\bmuestrame\b",
    r"\banaliza\b",
    r"\banalizar\b",
    r"\bcompara\b",
    r"\bcomparar\b",
    r"\bel historico del\b",
    r"\bel historico de\b",
    r"\bhistorico del\b",
    r"\bhistorico de\b",
    r"\bdatos del\b",
    r"\bdatos de\b",
    r"\bcotizacion del\b",
    r"\bcotizacion de\b",
    r"\bprecio del\b",
    r"\bprecio de\b",
    r"\bevolucion del\b",
    r"\bevolucion de\b",
    r"\bserie de\b",
    r"\braw\b",
]


TIME_PATTERNS_FOR_REMOVAL = [
    r"\b(1m|2m|5m|15m|30m|60m|90m|1h|1d|5d|1wk|1w|1mo|3mo)\b",
    r"\bdiari[oa]s?\b",
    r"\bsemanales?\b",
    r"\bmensuales?\b",
    r"\bultim[oa]s?\s+\d+\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
    r"\ben\s+\d+\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
    r"\bde\s+\d+\s+(dia|dias|semana|semanas|mes|meses|ano|anos|año|años)\b",
    r"\b(?:desde|hasta|a)\s+\d{4}(?!-)\b",
    r"\b(?:desde|hasta|a)\s+(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2}|\d{2}-\d{2}-\d{4})\b",
    r"\b(?:20\d{2}|19\d{2})-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])\b",
    r"\b(?:0[1-9]|[12]\d|3[01])/(?:0[1-9]|1[0-2])/(?:20\d{2}|19\d{2})\b",
    r"\b(?:20\d{2}|19\d{2})/(?:0[1-9]|1[0-2])/(?:0[1-9]|[12]\d|3[01])\b",
    r"\b(?:0[1-9]|[12]\d|3[01])-(?:0[1-9]|1[0-2])-(?:20\d{2}|19\d{2})\b",
    r"\b(ytd|year to date|ano hasta la fecha|este ano|max|maximo)\b",
]

LEADING_FILLERS = {"el", "la", "los", "las", "de", "del", "para", "sobre", "al", "un", "una"}
TRAILING_FILLERS = {"a", "de", "del", "para", "sobre", "en"}


def remove_patterns(text: str, patterns: list[str]) -> str:
    out = text
    for p in patterns:
        out = re.sub(p, " ", out, flags=re.IGNORECASE)
    return normalize_spaces(out)


def clean_asset_chunk(text: str) -> str:
    text = normalize_spaces(text.strip(" ,;:.-"))
    parts = text.split()
    while parts and normalize_text(parts[0]) in LEADING_FILLERS:
        parts.pop(0)
    while parts and normalize_text(parts[-1]) in TRAILING_FILLERS:
        parts.pop()
    return normalize_spaces(" ".join(parts))


def extract_asset_queries(message: str) -> list[str]:
    original = message.strip()

    # Caso simple: si hay algo entre comillas, se usa como pista fuerte
    quoted = re.findall(r'"([^"]+)"|\'([^\']+)\'', original)
    quoted = [normalize_text(q1 or q2) for q1, q2 in quoted if (q1 or q2)]
    if quoted:
        return [clean_asset_chunk(q) for q in quoted if clean_asset_chunk(q)]

    text = normalize_text(original)
    text = remove_patterns(text, COMMAND_PATTERNS)
    text = remove_patterns(text, TIME_PATTERNS_FOR_REMOVAL)

    # Limpieza adicional
    text = re.sub(r"[¿?¡!]", " ", text)
    text = normalize_spaces(text)

    # Divide posibles comparaciones
    chunks = re.split(r"\s*(?:,|;|\by\b|\be\b|\bvs\.?\b|\bcontra\b|\bfrente a\b)\s*", text, flags=re.IGNORECASE)
    chunks = [clean_asset_chunk(c) for c in chunks]
    chunks = [c for c in chunks if c]

    # Si no se detecta nada, usar el texto normalizado como último intento
    return chunks or [normalize_text(original)]


# =========================
# 6) Resolución de activos con yfinance
# =========================

ALLOWED_QUOTE_TYPES = {
    "EQUITY",
    "ETF",
    "INDEX",
    "MUTUALFUND",
    "FUTURE",
    "CURRENCY",
    "CRYPTOCURRENCY",
}


def _records_from_any(obj: Any) -> list[dict[str, Any]]:
    if obj is None:
        return []

    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")

    if isinstance(obj, list):
        out = []
        for item in obj:
            if isinstance(item, dict):
                out.append(item)
            elif hasattr(item, "__dict__"):
                out.append(vars(item))
        return out

    if isinstance(obj, dict):
        if "quotes" in obj and isinstance(obj["quotes"], list):
            return [x for x in obj["quotes"] if isinstance(x, dict)]
        return [obj]

    if hasattr(obj, "to_dict"):
        try:
            data = obj.to_dict()
            if isinstance(data, list):
                return [x for x in data if isinstance(x, dict)]
            if isinstance(data, dict):
                return [data]
        except Exception:
            pass

    return []


def score_candidate(query: str, candidate: dict[str, Any]) -> float:
    q = normalize_text(query)
    symbol = normalize_text(str(candidate.get("symbol", "")))
    shortname = normalize_text(str(candidate.get("shortname", "")))
    longname = normalize_text(str(candidate.get("longname", "")))
    qt = str(candidate.get("quoteType", "")).upper()

    score = 0.0

    if symbol == q:
        score += 1.0
    elif q and q in symbol:
        score += 0.55

    for name in (shortname, longname):
        if not name:
            continue
        if name == q:
            score += 0.95
        elif name.startswith(q):
            score += 0.65
        elif q in name:
            score += 0.45

    if qt in ALLOWED_QUOTE_TYPES:
        score += 0.25

    # Priorizamos activos negociables "reales" sobre resultados raros
    if qt == "EQUITY":
        score += 0.12
    elif qt in {"INDEX", "FUTURE", "CURRENCY", "CRYPTOCURRENCY", "ETF"}:
        score += 0.10

    return score


class YFinanceAssetResolver:
    def __init__(self, alias_to_ticker: Optional[dict[str, str]] = None):
        self.alias_to_ticker = alias_to_ticker or ALIAS_TO_TICKER

    def resolve(self, asset_query: str) -> ResolvedAsset:
        q_raw = normalize_spaces(asset_query)
        q_norm = normalize_text(q_raw)

        # 1) Alias manual directo
        if q_norm in self.alias_to_ticker:
            ticker = self.alias_to_ticker[q_norm]
            return ResolvedAsset(
                user_text=q_raw,
                ticker=ticker,
                name=q_raw,
                quote_type="ALIAS",
                exchange=None,
                source="alias_map",
                confidence=0.99,
            )

        # 2) Si parece ticker directo, usarlo como candidato fuerte.
        # Evitamos tratar nombres completos de empresa como ticker ("Nvidia" != "NVDA").
        direct_ticker_pattern = r"^[A-Za-z\^][A-Za-z0-9=\-.\^]{0,20}$"
        direct_candidate = None
        q_clean = q_raw.strip()
        looks_like_direct_ticker = (
            re.fullmatch(direct_ticker_pattern, q_clean) is not None
            and (
                q_clean.isupper()
                or bool(re.search(r"[\^=\-.0-9]", q_clean))
                or (len(q_clean) <= 5 and q_clean.upper() == q_clean)
            )
        )
        if looks_like_direct_ticker:
            direct_candidate = ResolvedAsset(
                user_text=q_raw,
                ticker=q_clean.upper(),
                name=q_clean,
                quote_type="DIRECT",
                exchange=None,
                source="direct_ticker_pattern",
                confidence=0.75,
            )

        # 3) Intento con Search
        search_candidates: list[dict[str, Any]] = []
        try:
            if hasattr(yf, "Search"):
                result = yf.Search(q_raw, max_results=10)
                search_candidates.extend(_records_from_any(getattr(result, "quotes", None)))
        except Exception:
            pass

        # 4) Intento con Lookup
        try:
            if hasattr(yf, "Lookup"):
                lookup = yf.Lookup(q_raw)
                for attr in ["all", "stock", "etf", "index", "future", "currency", "cryptocurrency", "mutualfund"]:
                    if hasattr(lookup, attr):
                        search_candidates.extend(_records_from_any(getattr(lookup, attr)))
        except Exception:
            pass

        if search_candidates:
            # Deduplicado por símbolo
            unique = {}
            for cand in search_candidates:
                sym = str(cand.get("symbol", "")).strip()
                if sym and sym not in unique:
                    unique[sym] = cand

            ranked = sorted(
                unique.values(),
                key=lambda x: score_candidate(q_raw, x),
                reverse=True,
            )
            best = ranked[0]
            return ResolvedAsset(
                user_text=q_raw,
                ticker=str(best.get("symbol", "")).strip(),
                name=best.get("shortname") or best.get("longname") or q_raw,
                quote_type=best.get("quoteType"),
                exchange=best.get("exchange") or best.get("exchDisp"),
                source="yfinance_search_lookup",
                confidence=round(score_candidate(q_raw, best), 4),
            )

        # 5) Fallback a ticker directo si lo parecía
        if direct_candidate:
            return direct_candidate

        raise ValueError(f"No se pudo resolver el activo: {asset_query!r}")


# =========================
# 7) Agente principal
# =========================

class YFinanceRequestAgent:
    def __init__(
        self,
        resolver: Optional[YFinanceAssetResolver] = None,
        default_interval: str = "1d",
        default_auto_adjust: bool = False,
    ):
        self.resolver = resolver or YFinanceAssetResolver()
        self.default_interval = default_interval
        self.default_auto_adjust = default_auto_adjust

    def interpret(self, user_message: str) -> DownloadRequest:
        asset_queries = extract_asset_queries(user_message)
        start, end, period, assumptions = parse_dates_or_period(user_message)
        interval = parse_interval(user_message) or self.default_interval

        resolved_assets = [self.resolver.resolve(q) for q in asset_queries]

        draft = DownloadRequest(
            user_message=user_message,
            asset_queries=asset_queries,
            resolved_assets=resolved_assets,
            start=start,
            end=end,
            period=period,
            interval=interval,
            auto_adjust=self.default_auto_adjust,
            assumptions=assumptions.copy(),
        )

        days = estimate_days(draft)
        fixed_interval, interval_note = fix_interval_if_needed(draft.interval, days)
        draft.interval = fixed_interval
        if interval_note:
            draft.assumptions.append(interval_note)

        return draft

    def download(self, request: DownloadRequest) -> pd.DataFrame:
        kwargs = request.to_download_kwargs()
        raw = yf.download(**kwargs)
        return raw

    def export_request_and_csv(
        self,
        request: DownloadRequest,
        raw: pd.DataFrame,
        output_dir: str | Path = "exports",
        filename_prefix: Optional[str] = None,
    ) -> dict[str, Any]:
        outdir = Path(output_dir)
        outdir.mkdir(parents=True, exist_ok=True)

        tickers_part = "-".join(request.tickers)[:80]
        range_part = request.period or f"{request.start}_to_{request.end}"
        base = filename_prefix or f"{slugify(tickers_part)}__{slugify(range_part)}__{slugify(request.interval)}"

        csv_path = outdir / f"{base}.csv"
        meta_path = outdir / f"{base}.metadata.json"

        raw.to_csv(csv_path, index=True)

        metadata = {
            "request": request.model_dump(),
            "download_kwargs": request.to_download_kwargs(),
            "csv_path": str(csv_path.resolve()),
            "shape": list(raw.shape),
            "generated_at": datetime.utcnow().isoformat() + "Z",
        }

        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        return {
            "csv_path": csv_path,
            "metadata_path": meta_path,
            "metadata": metadata,
        }

    def run(
        self,
        user_message: str,
        output_dir: str | Path = "exports",
        filename_prefix: Optional[str] = None,
    ) -> dict[str, Any]:
        request = self.interpret(user_message)
        raw = self.download(request)
        export_info = self.export_request_and_csv(
            request=request,
            raw=raw,
            output_dir=output_dir,
            filename_prefix=filename_prefix,
        )
        return {
            "request": request,
            "raw": raw,
            **export_info,
        }


# =========================
# 8) Utilidades de presentación
# =========================

def display_request_summary(request: DownloadRequest) -> pd.DataFrame:
    rows = []
    for asset in request.resolved_assets:
        rows.append(
            {
                "consulta_usuario": asset.user_text,
                "ticker": asset.ticker,
                "nombre": asset.name,
                "tipo": asset.quote_type,
                "exchange": asset.exchange,
                "source": asset.source,
                "confidence": asset.confidence,
            }
        )
    return pd.DataFrame(rows)


def flatten_columns_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        flat = df.copy()
        flat.columns = [
            "__".join([str(x) for x in tup if str(x) != ""]).strip("_")
            for tup in flat.columns.to_flat_index()
        ]
        return flat
    return df.copy()



## Ejemplos de interpretación sin descargar todavía

Esta parte sirve para verificar cómo **entiende** el agente el mensaje antes de lanzar `yfinance`.


In [4]:

agent = YFinanceRequestAgent()

sample_queries = [
    "Cuánto ha crecido Nvidia en 5 años",
    "Descárgame el histórico del S&P 500 desde 2020",
    "Quiero el oro en 1 semana a 1h",
    "Compara Nvidia y AMD en 2 años",
    "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
]

for q in sample_queries:
    print("=" * 100)
    print("Consulta:", q)
    try:
        req = agent.interpret(q)
        print(req.model_dump_json(indent=2, ensure_ascii=False))
        display(display_request_summary(req))
    except Exception as e:
        print("ERROR:", e)


Consulta: Cuánto ha crecido Nvidia en 5 años
ERROR: No se pudo resolver el activo: 'nvidia'
Consulta: Descárgame el histórico del S&P 500 desde 2020
{
  "user_message": "Descárgame el histórico del S&P 500 desde 2020",
  "asset_queries": [
    "s&p 500"
  ],
  "resolved_assets": [
    {
      "user_text": "s&p 500",
      "ticker": "^GSPC",
      "name": "s&p 500",
      "quote_type": "ALIAS",
      "exchange": null,
      "source": "alias_map",
      "confidence": 0.99
    }
  ],
  "start": "2020-01-01",
  "end": "2026-03-13",
  "period": null,
  "interval": "1d",
  "group_by": "ticker",
  "auto_adjust": false,
  "threads": true,
  "progress": false,
  "assumptions": [],
  "needs_clarification": false
}


,consulta_usuario,ticker,nombre,tipo,exchange,source,confidence
0,s&p 500,^GSPC,s&p 500,ALIAS,None,alias_map,0.99


Consulta: Quiero el oro en 1 semana a 1h
{
  "user_message": "Quiero el oro en 1 semana a 1h",
  "asset_queries": [
    "oro"
  ],
  "resolved_assets": [
    {
      "user_text": "oro",
      "ticker": "GC=F",
      "name": "oro",
      "quote_type": "ALIAS",
      "exchange": null,
      "source": "alias_map",
      "confidence": 0.99
    }
  ],
  "start": "2026-03-05",
  "end": "2026-03-13",
  "period": null,
  "interval": "1h",
  "group_by": "ticker",
  "auto_adjust": false,
  "threads": true,
  "progress": false,
  "assumptions": [
    "Las semanas se transforman a start/end para mantener compatibilidad."
  ],
  "needs_clarification": false
}


,consulta_usuario,ticker,nombre,tipo,exchange,source,confidence
0,oro,GC=F,oro,ALIAS,None,alias_map,0.99


Consulta: Compara Nvidia y AMD en 2 años
ERROR: No se pudo resolver el activo: 'nvidia'
Consulta: Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31
{
  "user_message": "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
  "asset_queries": [
    "bitcoin"
  ],
  "resolved_assets": [
    {
      "user_text": "bitcoin",
      "ticker": "BTC-USD",
      "name": "bitcoin",
      "quote_type": "ALIAS",
      "exchange": null,
      "source": "alias_map",
      "confidence": 0.99
    }
  ],
  "start": "2024-01-01",
  "end": "2025-01-01",
  "period": null,
  "interval": "1d",
  "group_by": "ticker",
  "auto_adjust": false,
  "threads": true,
  "progress": false,
  "assumptions": [],
  "needs_clarification": false
}


,consulta_usuario,ticker,nombre,tipo,exchange,source,confidence
0,bitcoin,BTC-USD,bitcoin,ALIAS,None,alias_map,0.99



## Ejecución end-to-end: descargar y exportar CSV

Modifica `USER_MESSAGE` y ejecuta la celda.
El agente:
1. interpreta el mensaje;
2. resuelve los ticker(s);
3. descarga los datos con `yfinance`;
4. guarda un CSV en `exports/`.


In [5]:

USER_MESSAGE = "Cuánto ha crecido Nvidia en 5 años"
OUTPUT_DIR = "exports"

result = agent.run(USER_MESSAGE, output_dir=OUTPUT_DIR)

print("CSV guardado en:", result["csv_path"])
print("Metadata guardada en:", result["metadata_path"])
display(display_request_summary(result["request"]))
result["raw"].tail()


ValueError: No se pudo resolver el activo: 'nvidia'


## Opción: dejar exactamente el `raw` de `yfinance` o generar una versión plana

`raw.to_csv(...)` conserva la estructura de salida de `yfinance`.
Si luego quieres un CSV más cómodo para downstream, puedes aplanar columnas.


In [6]:

raw_df = result["raw"]
flat_df = flatten_columns_if_needed(raw_df)

print("Columnas originales:", raw_df.columns)
print("Columnas aplanadas:", flat_df.columns[:10])

flat_csv_path = Path(OUTPUT_DIR) / "flat_example.csv"
flat_df.to_csv(flat_csv_path, index=True)

print("CSV plano guardado en:", flat_csv_path.resolve())
flat_df.tail()


NameError: name 'result' is not defined


## Función de alto nivel para reutilizar en tu proyecto

Puedes copiar esta interfaz al pipeline de tu TFM.


In [ ]:

def first_agent_download_to_csv(
    user_message: str,
    output_dir: str | Path = "exports",
    auto_adjust: bool = False,
) -> dict[str, Any]:
    local_agent = YFinanceRequestAgent(default_auto_adjust=auto_adjust)
    return local_agent.run(user_message=user_message, output_dir=output_dir)


# Ejemplo:
# out = first_agent_download_to_csv("Compara Apple y Microsoft en 3 años")
# print(out["csv_path"])



## Limitaciones y siguientes mejoras

1. **La resolución de activos depende de Yahoo Finance** y de lo bien que `Search` / `Lookup` respondan.
2. Para consultas ambiguas como *"¿cuánto ha crecido Nvidia?"* se usa un **rango por defecto de 1 año**.
3. Para intervalos intradía en rangos largos, el agente degrada el intervalo a `1d`.
4. Si quieres acercarte aún más al diseño final del TFM, el siguiente paso natural es:
   - sustituir la parte heurística de `interpret(...)` por un **LLM con salida estructurada**;
   - mantener igual el resto: resolución de ticker, descarga y exportación.
